# 10 Import time series from Excel

<!-- navigation -->
[<kbd>← 01 Piezometer locations</kbd>](01%20Piezometer%20locations.ipynb) &nbsp; [<kbd>Overview</kbd>](README.md) &nbsp; [<kbd>11 Download BRO groundwater levels →</kbd>](11%20Download%20BRO%20groundwater%20levels.ipynb)

<div class="alert alert-block alert-danger" style="background-color: #f8d7da; color: #842029; border: 1px solid #f5c2c7; border-radius: 4px; padding: 10px 14px;">
<b>For Dutch users:</b> it is highly recommended to load your data with notebooks <a href="11%20Download%20BRO%20groundwater%20levels.ipynb">11</a> (BRO groundwater measurements) and <a href="12%20Download%20river%20levels%20%28Rijkswaterstaat%29.ipynb">12</a> (Rijkswaterstaat river levels). In that case, you can skip notebook 10.
</div>

**Required**, unless you download the series with notebooks 11 and 12 · minimal path: 00 → 01 → **10** → 20 → 51 → 52

Imports the heads and river levels from the time series Excel file into a pastastore per transect.

| | |
|---|---|
| **Input** | `timeseries.xlsx` (template), piezometers in `project_data.gpkg`, `loc_river` in the transects metadata |
| **Output** | pastastore per transect |

The time series file has one column per series: the piezometer filters (named as in `piezometers.xlsx`) and the river gauge (named as `loc_river` in the transects metadata). Times are without time zone, in Dutch winter time (UTC+1). Each transect gets its own pastastore with its piezometers and its river gauge.

In [ ]:
from pathlib import Path

import timflow_levee_toolkit as tlt

import pandas as pd
tlt.show_versions()

In [ ]:
## UPDATE FOR YOUR PROJECT: the project folder (with project_settings.toml and the data)
project_dir = Path.cwd().parent / "examples" / "papendrecht"

project = tlt.load_project(project_dir)
units = project.units
project

## 1. Read the time series

In [ ]:
timeseries, series_info = tlt.read_timeseries_excel(project.general_file("timeseries"))
print(f"{timeseries.shape[1]} series, {timeseries.index.min()} - {timeseries.index.max()}")
timeseries.describe().T

## 2. Import into the pastastore of each transect

In [ ]:
overwrite = False  # True: replace series that are already in the pastastore

for transect in project.transects():
    meta = project.transect_metadata(transect)
    piezometers = project.piezometers(transect).set_index("name")

    missing = [n for n in [*piezometers.index, meta["loc_river"]] if n not in timeseries.columns]
    if missing:
        print(f"{transect}: no time series for {missing}")
    names = [n for n in [*piezometers.index, meta["loc_river"]] if n in timeseries.columns]

    series_metadata = pd.concat([pd.DataFrame(piezometers.drop(columns="geometry")), series_info])
    pstore = project.open_pastastore(transect)
    added = tlt.import_timeseries(pstore, timeseries[names], metadata=series_metadata, overwrite=overwrite)
    print(f"{transect}: {len(added)} series added, {len(pstore.oseries_names)} in the pastastore")

## 3. Check

In [ ]:
for transect in project.transects():
    pstore = project.open_pastastore(transect)
    tlt.plot_select(pstore, title=transect, subset="all series", figsize=(10, 5), units=units)

project.status()

---

<!-- navigation -->
[<kbd>← 01 Piezometer locations</kbd>](01%20Piezometer%20locations.ipynb) &nbsp; [<kbd>Overview</kbd>](README.md) &nbsp; [<kbd>11 Download BRO groundwater levels →</kbd>](11%20Download%20BRO%20groundwater%20levels.ipynb)